# 🏥 Anesthesia CoPilot — Live Inference Demo

**Scenario:** Patient C — The False Alarm (Bumped Sensor Artifact)

- Minutes 0–10: Both vitals are **healthy**. System should stay STABLE.
- Minute 10+: PLETH sensor is bumped → instant flatline. **But CO2 remains perfect.**
- The Hemo-Scout CNN will fire a CRASH alarm.
- The Conflict Resolver Transformer should **suppress** the alarm — it's just a sensor artifact.

---

## Cell 1 — Load Everything Into Memory

Run this cell **before** the presentation starts. Everything will be cached in GPU memory.

In [ ]:
import torch
import torch.nn as nn
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import clear_output, display
from scipy.signal import butter, sosfiltfilt

matplotlib.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ══════════════════════════════════════════════════════════════
# 1. Mount Google Drive
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# ── Edit this path to your Drive folder ──
DRIVE_DIR = '/content/drive/MyDrive/Datasets'

# ══════════════════════════════════════════════════════════════
# 2. Define Model Architectures (SABOTAGED — no Dropout)
# ══════════════════════════════════════════════════════════════

class HemoScout1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(128, 1),
        )
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


class VentGuardian(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 1),
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x).squeeze(1)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MicroTransformer(nn.Module):
    def __init__(self, input_dim=2, d_model=32, n_heads=2, n_layers=1,
                 dim_feedforward=64, dropout=0.0, seq_len=60):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len + 10, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 1),
        )
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        x = self.classifier(x)
        return x.squeeze(1)


# ══════════════════════════════════════════════════════════════
# 3. Load Frozen Brains from Google Drive
# ══════════════════════════════════════════════════════════════
import os

hemo_model = HemoScout1DCNN().to(device)
hemo_model.load_state_dict(torch.load(os.path.join(DRIVE_DIR, 'hemo_demo.pth'), map_location=device))
hemo_model.eval()
print("✅ Hemo-Scout loaded")

vent_model = VentGuardian().to(device)
vent_model.load_state_dict(torch.load(os.path.join(DRIVE_DIR, 'vent_demo.pth'), map_location=device))
vent_model.eval()
print("✅ Vent-Guardian loaded")

transformer_model = MicroTransformer(d_model=32, n_heads=2, n_layers=1).to(device)
transformer_model.load_state_dict(torch.load(os.path.join(DRIVE_DIR, 'transformer_demo.pth'), map_location=device))
transformer_model.eval()
print("✅ Conflict Resolver loaded")

# ══════════════════════════════════════════════════════════════
# 4. Load & Pre-Process Patient Data
# ══════════════════════════════════════════════════════════════
demo_data = pd.read_csv(os.path.join(DRIVE_DIR, 'demo_patient_C_artifact.csv'))
total_rows = len(demo_data)
total_seconds = total_rows // 100
print(f"\n📂 Patient C loaded: {total_rows:,} rows → {total_seconds} seconds ({total_seconds/60:.1f} min)")

# Apply the SAME Butterworth filter used during training
def apply_butterworth(signal, cutoff_hz, fs=100, order=4):
    nyquist = fs / 2
    normalized_cutoff = cutoff_hz / nyquist
    sos = butter(order, normalized_cutoff, btype='low', output='sos')
    return sosfiltfilt(sos, signal)

# Clean raw signals (same pipeline as 02_clean_and_chunk.py)
pleth_raw = demo_data['PLETH'].ffill().bfill().fillna(0).values
co2_raw   = demo_data['CO2'].ffill().bfill().fillna(0).values

pleth_filtered = apply_butterworth(pleth_raw, cutoff_hz=20).astype(np.float32)
co2_filtered   = apply_butterworth(co2_raw,   cutoff_hz=5).astype(np.float32)

print(f"   PLETH filtered range: [{pleth_filtered.min():.4f}, {pleth_filtered.max():.4f}]")
print(f"   CO2   filtered range: [{co2_filtered.min():.4f}, {co2_filtered.max():.4f}]")

print("\n" + "="*55)
print("  🟢 SYSTEM READY. All models loaded and locked.")
print("  📺 Run the next cell to start the live demo.")
print("="*55)

## Cell 2 — 🔴 LIVE DEMO (Run During Presentation)

This cell simulates real-time patient monitoring. It will:
1. Chop the CSV into 1-second frames
2. Run each frame through the Tier-1 CNNs
3. Feed the last 60 seconds into the Tier-2 Transformer
4. Display a live dashboard with vitals graph + terminal UI

**⏸️ Note:** The artifact event happens at **minute 10**. Watch for the Hemo-Scout alarm and the Transformer's response.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  🔴 ANESTHESIA COPILOT — LIVE INFERENCE ENGINE
# ══════════════════════════════════════════════════════════════

# ── Config ──
PLAYBACK_SPEED = 0.3   # Seconds to wait between frames (0.3 = ~3x speed)
VITALS_WINDOW  = 30    # Show last N seconds of vitals on graph
TOTAL_SECONDS  = total_seconds  # Full file length

# ── UI Helpers ──
def draw_bar(prob, width=20):
    filled = int(prob * width)
    return f"[{'█' * filled}{'░' * (width - filled)}]"

def get_status_color(status):
    if 'CRITICAL' in status:
        return '#ff4444'
    elif 'SUPPRESSED' in status:
        return '#ffaa00'
    elif 'INITIALIZING' in status:
        return '#4488ff'
    else:
        return '#44cc44'

# ── History buffers ──
thought_buffer = []          # Transformer's 60-second sliding window
hemo_history = []            # For plotting
vent_history = []            # For plotting
transformer_history = []     # For plotting
pleth_display_buffer = []    # Raw waveform for display
co2_display_buffer = []      # Raw waveform for display

# ── Main loop ──
for current_second in range(TOTAL_SECONDS):
    # 1. Slice 1 second of filtered data (100 samples)
    start_idx = current_second * 100
    end_idx   = start_idx + 100

    if end_idx > len(pleth_filtered):
        break

    pleth_chunk = pleth_filtered[start_idx:end_idx]
    co2_chunk   = co2_filtered[start_idx:end_idx]

    # Store raw signal for display graph
    pleth_display_buffer.append(pleth_chunk.copy())
    co2_display_buffer.append(co2_chunk.copy())

    # Format for PyTorch [1, 1, 100]
    pleth_tensor = torch.tensor(pleth_chunk, dtype=torch.float32).view(1, 1, 100).to(device)
    co2_tensor   = torch.tensor(co2_chunk, dtype=torch.float32).view(1, 1, 100).to(device)

    # ────────────────────────────────────────
    # TIER 1: CNN Gut Reactions
    # ────────────────────────────────────────
    with torch.no_grad():
        hemo_prob = torch.sigmoid(hemo_model(pleth_tensor)).item()
        vent_prob = torch.sigmoid(vent_model(co2_tensor)).item()

    hemo_history.append(hemo_prob)
    vent_history.append(vent_prob)
    thought_buffer.append([hemo_prob, vent_prob])

    # ────────────────────────────────────────
    # TIER 2: Transformer Conflict Resolution
    # ────────────────────────────────────────
    if len(thought_buffer) < 60:
        system_status = "⏳ INITIALIZING MEMORY BUFFER..."
        transformer_decision = 0.0
    else:
        if len(thought_buffer) > 60:
            thought_buffer.pop(0)

        seq_tensor = torch.tensor(thought_buffer, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            transformer_decision = torch.sigmoid(transformer_model(seq_tensor)).item()

        # Decision Logic
        if transformer_decision > 0.85:
            if hemo_prob > 0.8 and vent_prob < 0.3:
                system_status = "🚨 CRITICAL: CARDIOVASCULAR CRASH PREDICTED"
            elif vent_prob > 0.8 and hemo_prob < 0.3:
                system_status = "🚨 CRITICAL: RESPIRATORY OBSTRUCTION PREDICTED"
            else:
                system_status = "🚨 CRITICAL: MULTI-SYSTEM FAILURE"
        else:
            if hemo_prob > 0.85 or vent_prob > 0.85:
                system_status = "🟡 ALARM SUPPRESSED: SENSOR ARTIFACT DETECTED"
            else:
                system_status = "🟢 STABLE: NO INTERVENTION REQUIRED"

    transformer_history.append(transformer_decision)

    # ────────────────────────────────────────
    # RENDER DASHBOARD
    # ────────────────────────────────────────
    clear_output(wait=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 6),
                             gridspec_kw={'height_ratios': [1, 1], 'width_ratios': [1, 1]})
    fig.patch.set_facecolor('#1a1a2e')

    # Determine display window for vitals
    display_start = max(0, current_second - VITALS_WINDOW + 1)
    display_range = range(display_start, current_second + 1)

    # ── Top-Left: PLETH Waveform ──
    ax1 = axes[0, 0]
    ax1.set_facecolor('#0d1117')
    # Show raw waveform samples for last few seconds
    wave_secs = min(5, current_second + 1)  # Show last 5 sec of waveform
    wave_start = max(0, current_second + 1 - wave_secs)
    wave_data = np.concatenate(pleth_display_buffer[wave_start:current_second + 1])
    wave_t = np.linspace(wave_start, current_second + 1, len(wave_data))
    ax1.plot(wave_t, wave_data, color='#00ff88', linewidth=0.8, alpha=0.9)
    ax1.set_title('PLETH (Arterial Pulse)', color='#00ff88', fontweight='bold', fontsize=10)
    ax1.set_ylabel('Amplitude', color='#888888', fontsize=8)
    ax1.tick_params(colors='#555555', labelsize=7)
    ax1.set_xlim(wave_start, wave_start + 5)
    ax1.grid(True, alpha=0.15, color='#333333')

    # ── Top-Right: CO2 Waveform ──
    ax2 = axes[0, 1]
    ax2.set_facecolor('#0d1117')
    co2_wave_data = np.concatenate(co2_display_buffer[wave_start:current_second + 1])
    co2_wave_t = np.linspace(wave_start, current_second + 1, len(co2_wave_data))
    ax2.plot(co2_wave_t, co2_wave_data, color='#ff6b6b', linewidth=0.8, alpha=0.9)
    ax2.set_title('CO₂ (Capnography)', color='#ff6b6b', fontweight='bold', fontsize=10)
    ax2.set_ylabel('mmHg', color='#888888', fontsize=8)
    ax2.tick_params(colors='#555555', labelsize=7)
    ax2.set_xlim(wave_start, wave_start + 5)
    ax2.grid(True, alpha=0.15, color='#333333')

    # ── Bottom-Left: CNN Probabilities over time ──
    ax3 = axes[1, 0]
    ax3.set_facecolor('#0d1117')
    t_axis = list(display_range)
    ax3.plot(t_axis, hemo_history[display_start:], color='#00ff88',
             linewidth=1.5, label='Hemo-Scout', alpha=0.9)
    ax3.plot(t_axis, vent_history[display_start:], color='#ff6b6b',
             linewidth=1.5, label='Vent-Guardian', alpha=0.9)
    ax3.set_ylim(-0.05, 1.05)
    ax3.axhline(y=0.5, color='#ffaa00', linestyle='--', alpha=0.4, linewidth=0.8)
    ax3.set_title('TIER 1 — CNN Expert Opinions', color='#cccccc', fontweight='bold', fontsize=10)
    ax3.set_ylabel('Crash Probability', color='#888888', fontsize=8)
    ax3.set_xlabel('Time (seconds)', color='#888888', fontsize=8)
    ax3.legend(loc='upper left', fontsize=7, facecolor='#1a1a2e', edgecolor='#333333',
               labelcolor='#cccccc')
    ax3.tick_params(colors='#555555', labelsize=7)
    ax3.grid(True, alpha=0.15, color='#333333')

    # ── Bottom-Right: Transformer Decision over time ──
    ax4 = axes[1, 1]
    ax4.set_facecolor('#0d1117')
    ax4.plot(t_axis, transformer_history[display_start:], color='#4488ff',
             linewidth=2, label='Conflict Resolver', alpha=0.9)
    ax4.axhline(y=0.85, color='#ff4444', linestyle='--', alpha=0.5, linewidth=1,
                label='Alert Threshold')
    ax4.axhline(y=0.5, color='#ffaa00', linestyle='--', alpha=0.3, linewidth=0.8)
    ax4.set_ylim(-0.05, 1.05)
    ax4.set_title('TIER 2 — Transformer Final Decision', color='#4488ff', fontweight='bold', fontsize=10)
    ax4.set_ylabel('System Risk Score', color='#888888', fontsize=8)
    ax4.set_xlabel('Time (seconds)', color='#888888', fontsize=8)
    ax4.legend(loc='upper left', fontsize=7, facecolor='#1a1a2e', edgecolor='#333333',
               labelcolor='#cccccc')
    ax4.tick_params(colors='#555555', labelsize=7)
    ax4.grid(True, alpha=0.15, color='#333333')

    plt.tight_layout(rect=[0, 0.08, 1, 0.93])
    fig.suptitle('ANESTHESIA COPILOT — PATIENT C (Sensor Artifact Scenario)',
                 color='white', fontsize=13, fontweight='bold')
    plt.show()

    # ── Terminal-style text dashboard ──
    minutes = current_second // 60
    seconds = current_second % 60
    total_min = TOTAL_SECONDS // 60
    total_sec = TOTAL_SECONDS % 60

    print("═" * 58)
    print("       ANESTHESIA COPILOT — LIVE INFERENCE ENGINE")
    print("═" * 58)
    print(f"  Time: {minutes:02d}:{seconds:02d} / {total_min:02d}:{total_sec:02d}")
    print()
    print("  [TIER 1] EXPERT OPINIONS (1-Second Window):")
    print(f"  → Hemo-Scout (Arterial):  {draw_bar(hemo_prob)} {hemo_prob*100:5.1f}% RISK")
    print(f"  → Vent-Guardian (Lungs):  {draw_bar(vent_prob)} {vent_prob*100:5.1f}% RISK")
    print()
    print("  [TIER 2] TRANSFORMER DECISION (60-Sec Context):")
    print(f"  → Final System Risk:      {draw_bar(transformer_decision)} {transformer_decision*100:5.1f}%")
    print()
    print(f"  >> STATUS: {system_status}")
    print("═" * 58)

    time.sleep(PLAYBACK_SPEED)

print("\n\n" + "═" * 58)
print("       ✅ SIMULATION COMPLETE")
print("═" * 58)

---

## What to explain during the demo

| Time | Event | What to say |
|---|---|---|
| **0:00 – 10:00** | Both vitals healthy | *"Both CNNs see normal waveforms. The Transformer agrees — the system stays green."* |
| **~10:00** | PLETH flatlines suddenly | *"The arterial sensor was physically bumped. The PLETH signal goes to zero. Notice the Hemo-Scout immediately fires a crash alarm…"* |
| **~10:00 – 10:30** | Hemo high, Vent low | *"…but look at the CO2 — it's perfectly normal. The patient is still breathing fine. This is a classic false alarm."* |
| **~11:00+** | Transformer stays low | *"The Conflict Resolver Transformer has learned to cross-reference both signals over a 60-second window. It sees that only ONE sensor is alarming, while the other is stable. It correctly suppresses the alarm."* |
| **End** | Wrap up | *"This is the core architecture — Go to NestJS to PyTorch, three tiers of AI working together to prevent alarm fatigue in the OR."* |